# Nettoyage des données
On part des fichiers bruts de `data/raw`.

In [1]:
import pandas as pd

prix = pd.read_csv("../data/raw/carbon_prices_daily.csv")
evenements = pd.read_csv("../data/raw/climate_events.csv")
emissions = pd.read_csv("../data/raw/co2_emissions_yearly.csv")
mix = pd.read_csv("../data/raw/energy_mix_yearly.csv")
temperature = pd.read_csv("../data/raw/temperature_anomaly_monthly.csv")

## 1. Doublons
Constat : aucun doublon repéré à l'exploration. On le revérifie. S'il y en a 0, on ne supprime rien.

In [2]:
print("prix :", prix.duplicated().sum())
print("evenements :", evenements.duplicated().sum())
print("emissions :", emissions.duplicated().sum())
print("mix :", mix.duplicated().sum())
print("temperature :", temperature.duplicated().sum())

prix : 0
evenements : 0
emissions : 0
mix : 0
temperature : 0


## 2. Types : les dates
Constat : `date` et `year_month` sont lues en texte, ce qui empêche tout calcul par période. On les convertit avec `to_datetime`.

In [3]:
print("Avant :", prix["date"].dtype)

prix["date"] = pd.to_datetime(prix["date"])
evenements["date"] = pd.to_datetime(evenements["date"])
temperature["year_month"] = pd.to_datetime(temperature["year_month"])

print("Après :", prix["date"].dtype)
print("Lignes converties :", len(prix), len(evenements), len(temperature))

Avant : str
Après : datetime64[us]
Lignes converties : 15866 50 2528


## 3. Types : les booléens
Constat : `is_policy`, `is_extreme_weather` et `is_disaster` valent 0 ou 1. Ce sont des vrais / faux, on les convertit avec `astype(bool)`.

In [4]:
evenements["is_policy"] = evenements["is_policy"].astype(bool)
evenements["is_extreme_weather"] = evenements["is_extreme_weather"].astype(bool)
evenements["is_disaster"] = evenements["is_disaster"].astype(bool)

print(evenements[["is_policy", "is_extreme_weather", "is_disaster"]].dtypes)

is_policy             bool
is_extreme_weather    bool
is_disaster           bool
dtype: object


## 4. Colonnes redondantes
Constat : `temp_anomaly_baseline` contient toujours la même valeur, elle n'apporte rien : on la supprime. `year` et `month` répètent la date, mais ils servent aux `groupby` : on les garde après avoir vérifié qu'ils sont cohérents avec la date.

In [5]:
print(temperature["temp_anomaly_baseline"].unique())
temperature = temperature.drop(columns=["temp_anomaly_baseline"])

print("year cohérent avec date (prix) :", (prix["date"].dt.year == prix["year"]).all())
print("year cohérent avec year_month (température) :", (temperature["year_month"].dt.year == temperature["year"]).all())

<StringArray>
['1951-1980']
Length: 1, dtype: str
year cohérent avec date (prix) : True
year cohérent avec year_month (température) : True


## 5. Valeurs manquantes
Constat : seule `co2_ppm` a des manquants. Le ppm n'est mesuré que pour la région « Global ». Décision : on n'impute pas, on garde les `NaN` et on utilisera les lignes « Global » pour la question 4.

In [6]:
print("co2_ppm manquants :", temperature["co2_ppm"].isna().sum(), "sur", len(temperature))
print(temperature.groupby("region")["co2_ppm"].count())

co2_ppm manquants : 2212 sur 2528
region
Arctic                   0
Asia                     0
Europe                   0
Global                 316
North_America            0
Northern_Hemisphere      0
Southern_Hemisphere      0
Tropics                  0
Name: co2_ppm, dtype: int64


In [7]:
# 6. Noms de régions

print("Lignes concernées :", temperature["region"].str.contains("_").sum())
temperature["region"] = temperature["region"].str.replace("_", " ")
print(sorted(temperature["region"].unique()))

Lignes concernées : 948
['Arctic', 'Asia', 'Europe', 'Global', 'North America', 'Northern Hemisphere', 'Southern Hemisphere', 'Tropics']


## 7. Année 2026 incomplète
Constat : tous les marchés et la température s'arrêtent en avril 2026. Décision : on garde ces lignes (les mois sont valides), mais on exclura 2026 des moyennes annuelles.

In [8]:
print("Prix en 2026 :", (prix["date"].dt.year == 2026).sum(), "lignes, dernier jour :", prix["date"].max())
print("Température en 2026 :", (temperature["year_month"].dt.year == 2026).sum(), "lignes, dernier mois :", temperature["year_month"].max())

Prix en 2026 : 430 lignes, dernier jour : 2026-04-30 00:00:00
Température en 2026 : 32 lignes, dernier mois : 2026-04-01 00:00:00


## 8. Contrôles de cohérence
Constat : les 8 parts du mix doivent faire environ 100 %. Pour les émissions, 49 lignes ont un `co2_per_capita_t` qui s'écarte de émissions / population : la population est arrondie à 0,1 million (écart maximal de 0,05 million, vérifié à l'exploration). Décision : on garde `co2_per_capita_t` et on ne la recalcule pas.

In [9]:
parts = ["coal_pct", "oil_pct", "gas_pct", "nuclear_pct",
         "hydro_pct", "solar_pct", "wind_pct", "other_renewables_pct"]
somme_parts = mix[parts].sum(axis=1)
print("Mix : lignes dont la somme est loin de 100 :", ((somme_parts < 99) | (somme_parts > 101)).sum())

ecart_per_capita = (emissions["co2_emissions_mt"] / emissions["population_millions"] - emissions["co2_per_capita_t"]).abs()
print("Émissions : lignes avec un écart per capita > 0.1 :", (ecart_per_capita > 0.1).sum())

Mix : lignes dont la somme est loin de 100 : 0
Émissions : lignes avec un écart per capita > 0.1 : 49


## 9. Valeurs aberrantes : les prix du carbone
Constat : la règle des 1,5 écart interquartile signale des prix sur l'EU ETS et RGGI. On regarde à quelles années ils correspondent avant de décider. Aberrant ne veut pas dire faux.

In [10]:
prix_eu = prix[prix["market"] == "EU_ETS"]
q1 = prix_eu["price"].quantile(0.25)
q3 = prix_eu["price"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
eu_tres_chers = prix_eu[prix_eu["price"] > borne_haute]
print("EU_ETS : prix au-dessus de", round(borne_haute, 1), ":", len(eu_tres_chers))
print(eu_tres_chers["year"].value_counts().sort_index())

EU_ETS : prix au-dessus de 80.2 : 544
year
2021      5
2022    190
2023    250
2024     13
2025     14
2026     72
Name: count, dtype: int64


In [11]:
prix_rggi = prix[prix["market"] == "RGGI"]
q1 = prix_rggi["price"].quantile(0.25)
q3 = prix_rggi["price"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
rggi_tres_chers = prix_rggi[prix_rggi["price"] > borne_haute]
print("RGGI : prix au-dessus de", round(borne_haute, 1), ":", len(rggi_tres_chers))
print(rggi_tres_chers["year"].value_counts().sort_index())

RGGI : prix au-dessus de 20.2 : 296
year
2024     56
2025    203
2026     37
Name: count, dtype: int64


Décision : pour l'EU ETS, 440 des 544 prix aberrants datent de 2022-2023 (hausse du marché pendant la crise énergétique). Pour RGGI, les 296 prix se situent entre 2024 et 2026, juste au-dessus de la borne (maximum de 22,22 $). Ce sont des valeurs réelles, pas des erreurs de saisie : on les garde.

## 10. Enregistrement des fichiers propres

In [12]:
prix.to_csv("../data/processed/carbon_prices_propre.csv", index=False)
evenements.to_csv("../data/processed/climate_events_propre.csv", index=False)
emissions.to_csv("../data/processed/co2_emissions_propre.csv", index=False)
mix.to_csv("../data/processed/energy_mix_propre.csv", index=False)
temperature.to_csv("../data/processed/temperature_anomaly_propre.csv", index=False)

controle = pd.read_csv("../data/processed/carbon_prices_propre.csv", parse_dates=["date"])
print(controle.shape, controle["date"].dtype)

(15866, 5) datetime64[us]


## Journal de nettoyage

| Quoi | Pourquoi | Lignes touchées |
|---|---|---|
| Doublons : aucun supprimé | 0 doublon dans les 5 fichiers | 0 |
| `date` et `year_month` en vraies dates | Permettre les calculs par période | 15 866 (prix), 50 (événements), 2 528 (température) |
| `is_*` converties en booléens | Ce sont des vrai / faux | 50 |
| `temp_anomaly_baseline` supprimée | Valeur constante, aucune information | 2 528 (colonne) |
| `co2_ppm` : manquants conservés | Mesuré seulement pour « Global » | 2 212 sur 2 528 |
| `North_America` devient `North America` (idem pour les deux hémisphères) | Harmoniser avec les émissions avant jointure | 948 |
| 2026 signalée, exclue des moyennes annuelles | Année incomplète (jusqu'au 30/04/2026) | 430 (prix), 32 (température) |
| Écarts `co2_per_capita_t` conservés | Population arrondie à 0,1 million | 49 |
| Prix aberrants conservés | Hausses réelles du marché | 544 (EU ETS), 296 (RGGI) |
| Mix : parts cohérentes | Somme proche de 100 % | 0 anomalie |